
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Online Evaluation with Multi-Turn Conversations
#### Duration: 45 minutes

## Overview

In this lab, you are the ML engineer responsible for a production travel agent. The setup cell generates multi-turn conversation traces tagged with session IDs. Your job is to deploy multi-turn evaluation judges, verify they are running, and tune their sampling rates for production.

**This lab assumes familiarity with online evaluation architecture, scorer types, and sampling strategies.**

## Learning Objectives

1. Configure UC-backed trace monitoring by tagging the experiment with a SQL warehouse ID
2. Configure multi-turn judges and verify they are active using `list_scorers()`
3. Inspect and operate the trace metrics computation job
4. Create custom scorers with the `@scorer` decorator and appropriate aggregations
5. Manage the scorer lifecycle: stop, update, and delete

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Lab Structure</strong>
<p style="margin: 8px 0 0 0; color: #333;">Each section follows a <strong>build → verify</strong> pattern. TODO cells contain code you write; instructional cells guide you through the UI with expandable steps. Expand the <strong>Answer</strong> dropdown if you get stuck.</p>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
<li><strong>SQL Warehouse</strong>: Required for UC-backed trace monitoring. The monitoring job queries UC trace tables through this warehouse. You need <strong>CAN USE</strong> permission on the warehouse.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>UC Trace Table Permissions:</strong> You need explicit <strong>MODIFY</strong> and <strong>SELECT</strong> on the UC trace tables. <code>ALL_PRIVILEGES</code> is not sufficient.</p>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Setup

In [0]:
%run ./Includes/Classroom-Setup-06-Lab

### A1. Configure UC-Backed Trace Monitoring

The setup cell already configured UC trace storage for this experiment. Your traces are stored in four Delta tables (`traces_otel_spans`, `traces_otel_annotations`, `traces_otel_logs`, `traces_otel_metrics`) in your catalog and schema.

Before registering any scorers, you must also tag the experiment with a SQL warehouse ID. The trace metrics computation job runs **separately** from your notebook process and queries the UC trace tables through this warehouse. It reads the warehouse ID from the `mlflow.monitoring.sqlWarehouseId` experiment tag. Without it, the monitoring job fails.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">This Must Come First</strong>
<p style="margin: 8px 0 0 0; color: #333;">Set the SQL warehouse ID on the experiment <strong>before</strong> you register any scorers. If you skip this step, monitoring jobs will fail with an error saying the <code>mlflow.monitoring.sqlWarehouseId</code> tag is missing.</p>
<p style="margin: 8px 0 0 0; color: #333;"><strong>Important:</strong> Setting the <code>MLFLOW_TRACING_SQL_WAREHOUSE_ID</code> environment variable in your notebook does <strong>not</strong> work as a substitute. It only applies to the current process. The monitoring job runs separately and reads the ID from the experiment tag.</p>
</div>
</div>

**TODO:**
1. Import `set_databricks_monitoring_sql_warehouse_id` from `mlflow.tracing`
2. Call it with `sql_warehouse_id` (from setup) and optionally the experiment ID

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Step 1: Import set_databricks_monitoring_sql_warehouse_id from mlflow.tracing
# Step 2: Call it with sql_warehouse_id (provided by setup)
#
# Note: The experiment is already set from the setup cell.
#       The monitoring job reads the warehouse ID from the experiment tag,
#       NOT from environment variables.
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> A1. Configure UC-Backed Trace Monitoring
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.tracing import set_databricks_monitoring_sql_warehouse_id
<span></span>
set_databricks_monitoring_sql_warehouse_id(
    sql_warehouse_id=sql_warehouse_id,
)</div>
</div>

</div>
</details>

## B. Multi-Turn Judges

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">Users have reported that the agent sometimes gives incomplete answers or takes too many turns to resolve simple questions. You need multi-turn evaluation judges to detect these patterns across entire conversations.</div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Experimental</strong>
<p style="margin: 8px 0 0 0; color: #333;">Multi-turn evaluation is Experimental. Its API and behavior might change in future releases.</p>
</div>
</div>
</div>

### B1. Create and Deploy Conversation Judges

The user complaints describe two distinct problems: **incomplete answers** (the agent doesn't fully resolve the request) and **too many turns** (users have to repeat themselves). Single-turn safety checks can't catch these because they evaluate each response in isolation. You need multi-turn judges that analyze the full conversation thread to detect these patterns. Each judge below targets a specific dimension of the problem:

<div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 14px; margin: 18px 0; font-family: sans-serif;">
  <div style="margin: 0 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #2272B4;"></div>
    <div style="font-size: 15px; font-weight: 700; color: #0b2026; font-family: monospace; overflow-wrap: anywhere;">ConversationCompleteness</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Did the agent fully resolve the request?</div>
  </div>
  <div style="margin: 0 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>
    <div style="font-size: 15px; font-weight: 700; color: #0b2026; font-family: monospace; overflow-wrap: anywhere;">UserFrustration</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Is the user showing signs of frustration?</div>
  </div>
  <div style="margin: 0 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">
    <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #00A972;"></div>
    <div style="font-size: 15px; font-weight: 700; color: #0b2026; font-family: monospace; overflow-wrap: anywhere;">ConversationalSafety</div>
    <div style="font-size: 14px; color: #5A6F77; margin-top: 6px; line-height: 1.5;">Are there safety concerns across the session?</div>
  </div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">What happens when you call .start()</strong>
<p style="margin: 8px 0 0 0; color: #333;">Calling <code>.register()</code> and <code>.start()</code> activates <strong>online quality monitoring</strong> for your experiment. Databricks creates a background <strong>trace metrics computation job</strong> that continuously:</p>
<ul style="margin: 8px 0 0 16px; color: #333;">
<li>Pulls new traces from the experiment</li>
<li>Groups them into conversations via <code>mlflow.trace.session</code> metadata</li>
<li>Runs your scorers against each conversation</li>
<li>Logs the resulting assessments back as feedback on the traces</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;">This job is created automatically the first time you call <code>.start()</code>. It keeps running as long as you have active scorers (<code>sample_rate > 0</code>).</p>
</div>
</div>

<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #e65100; font-size: 1.1em;">Monitoring Job Identity</strong>
<p style="margin: 8px 0 0 0; color: #333;">The monitoring job runs as <strong>the user who registers the first scorer</strong>. That person's permissions on the SQL warehouse and UC trace tables determine what the job can access. In a shared workspace, make sure this user has CAN EDIT on the experiment and CAN USE on the warehouse.</p>
</div>
</div>

**TODO:**
1. Import all three judges from `mlflow.genai.scorers`, create each with `model=judge_model` (the course judge model service), register them with unique names, and start them.
    - Safety is critical: **sample_rate=1.0**
    - Completeness and frustration are expensive LLM judges: **sample_rate=0.5**

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Step 1: Import the three multi-turn judges + ScorerSamplingConfig
#         from mlflow.genai.scorers
# Step 2: Use mlflow.set_experiment() with mlflow_experiment_path
# Step 3: Instantiate each with model=judge_model, register (name= must be
#         unique), and start it
#
# Safety:       sample_rate=1.0
# Completeness: sample_rate=0.5
# Frustration:  sample_rate=0.5
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B1. Create and Deploy Conversation Judges
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">import mlflow
mlflow.set_experiment(mlflow_experiment_path)
<span></span>
from mlflow.genai.scorers import (
    ConversationCompleteness,
    UserFrustration,
    ConversationalSafety,
    ScorerSamplingConfig,
)
<span></span>
completeness = ConversationCompleteness(model=judge_model).register(name="conversation_completeness")
completeness = completeness.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5),
)
<span></span>
frustration = UserFrustration(model=judge_model).register(name="user_frustration")
frustration = frustration.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5),
)
<span></span>
safety = ConversationalSafety(model=judge_model).register(name="conversational_safety")
safety = safety.start(
    sampling_config=ScorerSamplingConfig(sample_rate=1.0),
)</div>
</div>

</div>
</details>

### B2. Add a Custom Judge via the UI

Before running the job, add one more judge, this time through the UI instead of code. This demonstrates that you can mix programmatic and UI-created judges on the same experiment. The job will evaluate all active judges in a single run.

Click each step below to expand the instructions.

<br></br>

<div style="max-width: 900px; margin: 0 auto; padding: 4px 6px; box-sizing: border-box; font-family: sans-serif;">

<style>

.lab-acc-item { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); margin-bottom:10px; overflow:hidden; transition:border-color 0.2s, box-shadow 0.2s; }
.lab-acc-item.open { border-color:#1B5162; box-shadow:0 2px 10px rgba(27,49,57,.10); }
.lab-acc-header { display:flex; align-items:center; gap:14px; padding:14px 18px; cursor:pointer; user-select:none; background:#fff; transition:background 0.15s; }
.lab-acc-header:hover { background:#F9F7F4; }
.lab-acc-num { width:30px; height:30px; border-radius:50%; background:#EEEDE9; color:#5A6F77; font-size:12.5pt; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0; transition:background 0.2s, color 0.2s; }
.lab-acc-item.open .lab-acc-num { background:#1B5162; color:#fff; }
.lab-acc-summary { flex:1; font-size:13.5pt; font-weight:700; color:#0b2026; }
.lab-acc-chevron { font-size:12pt; color:#9AABB1; transition:transform 0.25s, color 0.2s; }
.lab-acc-item.open .lab-acc-chevron { transform:rotate(180deg); color:#1B5162; }
.lab-acc-body { max-height:0; opacity:0; overflow:hidden; transition:max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding:0 18px 0 62px; font-size:14pt; line-height:1.7; color:#333; overflow-wrap:anywhere; }
.lab-acc-item.open .lab-acc-body { max-height:2000px; opacity:1; padding:2px 18px 18px 62px; }

</style>

<div id="lab-acc-b2"></div>

</div>

<script>
var LAB_B2 = [
  {
    summary: 'Navigate to the experiment',
    detail: 'Navigate to <strong>Experiments</strong> in the left menu. Search for and click on <strong>evaluation-and-monitoring-training</strong>.'
  },
  {
    summary: 'Open the Scorers panel',
    detail: 'In your experiment, click <strong>Scorers</strong> under <strong>Evaluation</strong>, then click <strong>+ New scorer</strong> to open the <strong>Create LLM scorer</strong> panel.'
  },
  {
    summary: 'Configure the scorer',
    detail: `In the <strong>LLM scorer</strong> dropdown, scroll to <strong>Session-level scores</strong> and select <strong>Conversational guidelines</strong>. Set <strong>Name</strong> to <strong>professional</strong> (it cannot be changed after creation). Leave <strong>Model</strong> on <strong>Databricks default</strong>. In the guidelines field, paste the text below:
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Guidelines</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">The interaction must be professional.</div>
</div>
Toggle on <strong>Auto-run on new traces</strong> at the bottom of the panel, and make sure <strong>Sample rate</strong> is set to 100%.`
  },
  {
    summary: 'Create',
    detail: 'Click <strong>Create Scorers</strong> at the bottom right. This will populate your experiment with a new scorer called <strong>professional</strong>.'
  }
];

(function() {
  var container = document.getElementById('lab-acc-b2');
  LAB_B2.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab-acc-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab-acc-header" onclick="labToggleB2(' + i + ')">'
      + '<div class="lab-acc-num">' + (i + 1) + '</div>'
      + '<div class="lab-acc-summary">' + item.summary + '</div>'
      + '<div class="lab-acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab-acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function labToggleB2(idx) {
  var items = document.querySelectorAll('#lab-acc-b2 .lab-acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### B3. Add a Custom Scorer in Code

The built-in judges evaluate quality dimensions like safety and completeness, but your team also wants to track a basic metric: **response length**. Overly long responses slow down users and increase token costs; overly short ones may indicate the agent is not being helpful. A custom scorer lets you measure this programmatically and get mean/min/max aggregations across all evaluated traces.

This demonstrates three ways to deploy scorers: out-of-the-box (B1), via the UI (B2), and custom code (B3). All will run together when the job executes.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Serialization Rules</strong>
<p style="margin: 8px 0 0 0; color: #333;">Custom scorers run remotely. All <code>import</code> statements must be <strong>inside the function body</strong>. Return consistent types. Handle missing data gracefully.</p>
</div>
</div>

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Create a @scorer function that:
#   - Decodes outputs with json.loads while it is a string
#   - Measures the length of outputs["response"] (or of outputs itself)
#   - Has aggregations=["mean", "min", "max"]
# Then register as "response_length" and start with sample_rate=0.5
#
# Hint: from mlflow.genai.scorers import scorer, ScorerSamplingConfig
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B3. Custom Scorer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import scorer, ScorerSamplingConfig
<span></span>
@scorer(aggregations=["mean", "min", "max"])
def response_length(outputs):
    """Measure response length in characters"""
    import json
    # Traces stored in Unity Catalog can pass outputs as JSON text
    while isinstance(outputs, str):
        try:
            outputs = json.loads(outputs)
        except ValueError:
            break
    if isinstance(outputs, dict):
        outputs = outputs.get("response", "")
    return len(str(outputs))
<span></span>
length_scorer = response_length.register(name="response_length")
length_scorer = length_scorer.start(
    sampling_config=ScorerSamplingConfig(sample_rate=0.5)
)</div>
</div>

</div>
</details>

### B4. Verify Sessions and Run the Job

All five scorers are now deployed: four judges (three built-in judges from B1 and the <strong>professional</strong> judge from B2) and one custom code scorer (B3). Verify the traces exist and run the job once to evaluate everything together. Click each step to expand.

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Default Conversation Timing</strong>
<p style="margin: 8px 0 0 0; color: #333;">Multi-turn judges run after a conversation is considered complete. By default, no new traces with that session ID for <strong>5 minutes</strong>. Make sure the setup script was run at least 5 minutes ago.</p>
</div>
</div>

<br></br>

<div style="max-width: 900px; margin: 0 auto; padding: 4px 6px; box-sizing: border-box; font-family: sans-serif;">

<style>

.lab-acc-item { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); margin-bottom:10px; overflow:hidden; transition:border-color 0.2s, box-shadow 0.2s; }
.lab-acc-item.open { border-color:#1B5162; box-shadow:0 2px 10px rgba(27,49,57,.10); }
.lab-acc-header { display:flex; align-items:center; gap:14px; padding:14px 18px; cursor:pointer; user-select:none; background:#fff; transition:background 0.15s; }
.lab-acc-header:hover { background:#F9F7F4; }
.lab-acc-num { width:30px; height:30px; border-radius:50%; background:#EEEDE9; color:#5A6F77; font-size:12.5pt; font-weight:700; display:flex; align-items:center; justify-content:center; flex-shrink:0; transition:background 0.2s, color 0.2s; }
.lab-acc-item.open .lab-acc-num { background:#1B5162; color:#fff; }
.lab-acc-summary { flex:1; font-size:13.5pt; font-weight:700; color:#0b2026; }
.lab-acc-chevron { font-size:12pt; color:#9AABB1; transition:transform 0.25s, color 0.2s; }
.lab-acc-item.open .lab-acc-chevron { transform:rotate(180deg); color:#1B5162; }
.lab-acc-body { max-height:0; opacity:0; overflow:hidden; transition:max-height 0.3s ease, opacity 0.25s ease, padding 0.25s ease; padding:0 18px 0 62px; font-size:14pt; line-height:1.7; color:#333; overflow-wrap:anywhere; }
.lab-acc-item.open .lab-acc-body { max-height:2000px; opacity:1; padding:2px 18px 18px 62px; }

</style>

<div id="lab-acc-b4"></div>

</div>

<script>
var LAB_B4 = [
  {
    summary: 'Verify sessions in the Experiment UI',
    detail: 'Navigate to <strong>Experiments</strong> in the left menu. Search for and click on <strong>evaluation-and-monitoring-training</strong>. Click on the <strong>Traces</strong> tab, then select <strong>Sessions</strong> in the Traces/Sessions switcher to see the session-level view. Note that your sessions have been populated. Click the dropdown next to a session to view its 3 turns.'
  },
  {
    summary: 'Find the Trace Metrics Computation Job',
    detail: 'Navigate to <strong>Jobs &amp; Pipelines</strong> and search for <em>Trace Metrics Computation Job</em>. Your job has the experiment ID prepended, e.g., <strong>[0123456789] Trace Metrics Computation Job</strong>. It runs on a fixed ~15-minute schedule. Your <code>sample_rate</code> controls how many traces are evaluated, not how often the job runs.<br><br><strong>UC-backed traces:</strong> This job executes scorer queries against the UC trace tables <em>through the SQL warehouse</em> you configured in section A1. If you see warehouse-related errors, verify you have CAN USE on the warehouse you configured in section A1.'
  },
  {
    summary: 'Pause the schedule and run manually',
    detail: 'For this lab, click <strong>Pause</strong> to stop the automatic schedule. Then click <strong>Run now</strong>. This single run will evaluate traces using all five active scorers: the three built-in multi-turn judges, the <strong>professional</strong> judge you created via the UI, and the <strong>response_length</strong> custom scorer.'
  },
  {
    summary: 'Inspect the job output with Genie Code',
    detail: `After the job completes, navigate to <strong>Runs</strong> and click the last successful run (green bar). Review the log output. Try <strong>Genie Code</strong> (top right next to your workspace name) and ask:
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Genie Code Prompt</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">What does this output tell me?</div>
</div>
Genie Code should respond with something like: <em>"In short, this output confirms that the agent monitoring metrics computation completed without errors for this run."</em> Read through the full response to understand what was evaluated.`
  },
  {
    summary: 'View scorer results on traces',
    detail: 'Navigate back to the experiment. On the <strong>Traces</strong> tab, select <strong>Sessions</strong> in the Traces/Sessions switcher to see session-level feedback. All four judges are session-level: <strong>ConversationCompleteness</strong>, <strong>UserFrustration</strong>, <strong>ConversationalSafety</strong>, and <strong>professional</strong> (its instructions use <code>{{conversation}}</code>). Their results appear on the first trace of each session, not on every trace. Switch to <strong>Traces</strong> in the switcher to see the per-trace <strong>response_length</strong> scores. Click on a trace, then click <strong>Assess</strong> at the top right to see the assessment results. Some sessions and traces may not have every result because of the sample rates (for example, <strong>response_length</strong> runs on 50% of traces).<br><br><strong>Take a moment to read the judge feedback carefully.</strong> Do you agree with the assessments? For example, does the completeness judge correctly flag Session 2 (vague flight answers) as incomplete? Does the frustration judge catch Session 3 (ignored cancellation request)? Evaluating whether you agree with the judges is a critical step in tuning your evaluation strategy.'
  }
];

(function() {
  var container = document.getElementById('lab-acc-b4');
  LAB_B4.forEach(function(item, i) {
    var div = document.createElement('div');
    div.className = 'lab-acc-item' + (i === 0 ? ' open' : '');
    div.innerHTML =
      '<div class="lab-acc-header" onclick="labToggleB4(' + i + ')">'
      + '<div class="lab-acc-num">' + (i + 1) + '</div>'
      + '<div class="lab-acc-summary">' + item.summary + '</div>'
      + '<div class="lab-acc-chevron">&#9660;</div>'
      + '</div>'
      + '<div class="lab-acc-body">' + item.detail + '</div>';
    container.appendChild(div);
  });
})();

function labToggleB4(idx) {
  var items = document.querySelectorAll('#lab-acc-b4 .lab-acc-item');
  items.forEach(function(el, i) {
    el.classList.toggle('open', i === idx && !el.classList.contains('open'));
  });
}
</script>

### B5. Verify Your Scorers Are Running

Confirm all five scorers were deployed correctly and are active. In production, this is the same audit step you would run to verify what is evaluating your traces and at what rates.

**TODO**: Use `list_scorers()` to enumerate all registered scorers. You should see 5 active scorers.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Import list_scorers from mlflow.genai.scorers
# List all scorers and print their name and sample rate
#
# Hint: scorer objects have .name and .sample_rate
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> B5. Verify Scorers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import list_scorers
<span></span>
scorers = list_scorers()
print(f"Found {len(scorers)} registered scorers:\n")
for s in scorers:
    status = "ACTIVE" if s.sample_rate &amp;gt; 0 else "STOPPED"
    print(f"  {s.name}: rate={s.sample_rate} [{status}]")</div>
</div>

</div>
</details>

## C. Sampling Strategy

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
<div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
<div style="font-size: 14pt; line-height: 1.5;">Your scorers have been running and the quality signals are stabilizing. Now you need to tune the sampling rates to balance cost and coverage for production.</div>
</div>

### C1. Update a Sampling Rate

After running the judges for a while, you notice the completeness scorer's results have stabilized: the pass rate hasn't changed in the last several evaluation cycles. Continuing to evaluate 50% of traces is wasting LLM calls. You can reduce the rate to 5% to save costs while still catching regressions. This is the typical production lifecycle: start high for fast feedback, dial back once the signal is stable.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Return value pattern</strong>
<p style="margin: 8px 0 0 0; color: #333;"><code>.update()</code> changes the scorer's configuration on the server but returns a <strong>new Python object</strong> reflecting the change. Always reassign: <code>scorer = scorer.update(...)</code></p>
</div>
</div>

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. Use get_scorer() to fetch "conversation_completeness"
# 2. Update its sample_rate to 0.05
# 3. Print the original and updated rates to show the return value pattern
# 4. List all scorers to see the full picture
#
# Hint: from mlflow.genai.scorers import get_scorer, ScorerSamplingConfig
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> C1. Update and Verify
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import get_scorer, list_scorers, ScorerSamplingConfig
<span></span>
# Fetch and update
original = get_scorer(name="conversation_completeness")
print(f"Original rate: {original.sample_rate}")
<span></span>
updated = original.update(
    sampling_config=ScorerSamplingConfig(sample_rate=0.05)
)
print(f"Updated rate:  {updated.sample_rate}")
<span></span>
# Full audit
print("\nAll scorers:")
for s in list_scorers():
    status = "ACTIVE" if s.sample_rate &amp;gt; 0 else "STOPPED"
    print(f"  {s.name}: rate={s.sample_rate} [{status}]")</div>
</div>

</div>
</details>

## D. Stop and Clean Up Scorers

When you are done with the scorers, stop them to pause evaluation, then delete the ones you no longer need.

### D1. Stop Deployed Scorers
You have confirmed the scorers work and tuned a sampling rate. Now stop all five scorers. In production, you would stop a scorer when you want to pause evaluation temporarily without losing the scorer's registration. The background job stops running once all scorers have `sample_rate = 0`.

**TODO**: Stop all five scorers: `completeness`, `frustration`, `safety`, `length_scorer` (from B3), and the `professional` judge (use `get_scorer(name="professional")` since it was created via the UI). Note that `.stop()` returns a new Python object. Always reassign (e.g., `safety = safety.stop()`) so your local variable reflects the updated state.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# Stop all 5 scorers:
# - completeness, frustration, safety (local variables from B1)
# - length_scorer (local variable from B3)
# - professional (use get_scorer(name="professional") to fetch it first)
# Note: .stop() returns a new Python object, so reassign to keep your variable current
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> D1. Stop Deployed Scorers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import get_scorer
<span></span>
# Programmatic scorers (B1)
completeness = completeness.stop()
print(f"Sample rate for completeness: {completeness.sample_rate}")
frustration = frustration.stop()
print(f"Sample rate for frustration: {frustration.sample_rate}")
safety = safety.stop()
print(f"Sample rate for safety: {safety.sample_rate}")
<span></span>
# Custom scorer (B3)
length_scorer = length_scorer.stop()
print(f"Sample rate for response_length: {length_scorer.sample_rate}")
<span></span>
# UI-created scorer (B2): fetch it with get_scorer, since there is no local variable
professional = get_scorer(name="professional")
professional = professional.stop()
print(f"Sample rate for professional: {professional.sample_rate}")</div>
</div>

</div>
</details>

### D2. Delete Scorers
In a shared workspace, abandoned scorers continue consuming resources. The background job keeps running, making LLM calls on every evaluation cycle. Deleting scorers you no longer need is important for cost management. Unlike `.stop()` (which pauses but keeps the scorer registered), `delete_scorer()` permanently removes it. Historical assessments already written to traces are preserved.

**TODO**: List all scorers, delete each one using `delete_scorer(name=...)`, and verify the list is empty.

In [0]:
# TODO ──────────────────────────────────────────────────────────────────
# 1. List all scorers using list_scorers()
# 2. Delete each one using delete_scorer(name=...)
# 3. Verify the list is empty
#
# Hint: from mlflow.genai.scorers import list_scorers, delete_scorer
# ──────────────────────────────────────────────────────────────────────


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> D2. Delete Scorers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;">

<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">from mlflow.genai.scorers import list_scorers, delete_scorer
<span></span>
scorers = list_scorers()
print(f"Deleting {len(scorers)} scorers:\n")
<span></span>
for s in scorers:
    print(f"  Deleting: {s.name}...", end=" ")
    delete_scorer(name=s.name)
    print("done")
<span></span>
# Verify
remaining = list_scorers()
print(f"\nRemaining scorers: {len(remaining)}")</div>
</div>

</div>
</details>

## Conclusion

1. **Multi-turn judges** evaluate entire conversations using the `mlflow.trace.session` metadata to group traces into sessions
2. **Custom scorers** use the `@scorer` decorator with all imports inline and support `aggregations` for numeric metrics
3. **Sampling rates** should be tuned by scorer type: 100% for safety, 5% for expensive LLM judges once stable, 30-50% during active development
4. **Return value pattern**: `.update()` and `.stop()` change the server state but return new Python objects -- always reassign so your local variable reflects the change
5. **`list_scorers()`** lets you audit what is running and at what rates
6. **UC-backed trace monitoring** requires setting the SQL warehouse ID on the experiment **before** registering any scorers — the monitoring job reads this from the `mlflow.monitoring.sqlWarehouseId` experiment tag, not from environment variables
7. **Permissions for UC traces**: explicit `MODIFY` and `SELECT` on trace tables (`ALL_PRIVILEGES` is not sufficient), `CAN USE` on the SQL warehouse, and `CAN EDIT` on the experiment

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
